# Imports & Constants

In [1]:
import pandas as pd
from pathlib import Path

In [ ]:
# paths
base_path = Path("/kaggle/input/playground-series-s5e12/")
train_csv_path = base_path / "train.csv"
test_csv_path = base_path / "test.csv"
submission_csv_path = base_path / "sample_submission.csv"

# load dataset
train_df = pd.read_csv(train_csv_path)
test_df = pd.read_csv(test_csv_path)

# others
RERUN = False# if true, reruns all visualisations and time-consuming operations
target_col = "diagnosed_diabetes"
RANDOM_SEED = 13

# Preprocess Data

In [3]:
# convert categorical string data to one-hot encoding
train_df = pd.get_dummies(train_df, drop_first=True, dtype=int)
test_df = pd.get_dummies(test_df, drop_first=True, dtype=int)

# Linear Models

In [7]:
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import (
    Lars,
    LassoLars,
    OrthogonalMatchingPursuit,
    BayesianRidge,
    ARDRegression,
    HuberRegressor,
    RANSACRegressor,
    LinearRegression,
    Ridge,
    Lasso,
    ElasticNet,
    TheilSenRegressor,
    PoissonRegressor,
    GammaRegressor,
    TweedieRegressor,
    SGDRegressor,
    QuantileRegressor,
)
from sklearn.model_selection import train_test_split
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.metrics import roc_auc_score
import pandas as pd

# ----- Your data -----
X, y = train_df.drop(columns=[target_col, "id"]), train_df[target_col]
X = X.reset_index(drop=True)
y = y.reset_index(drop=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED
)

# ----- Column-wise preprocessing -----
preprocess = ColumnTransformer(
    transformers=[
        ("standard_scaler", StandardScaler(), list(X.columns)),
    ],
    remainder="passthrough",
)


# ----- Wrapper that changes .score to ROC AUC and predict to 0/1 -----
class ROCThresholdWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, base_estimator, threshold: float = 0.5):
        self.base_estimator = base_estimator
        self.threshold = threshold

    def fit(self, X, y):
        # clone to avoid side effects
        self.estimator_ = clone(self.base_estimator)
        self.estimator_.fit(X, y)
        return self

    def _get_scores(self, X):
        # Prefer probabilities if available
        if hasattr(self.estimator_, "predict_proba"):
            return self.estimator_.predict_proba(X)[:, 1]
        # Or decision_function (e.g. linear SVM)
        if hasattr(self.estimator_, "decision_function"):
            return self.estimator_.decision_function(X)
        # Fallback: use raw predictions as scores (for regressors)
        return self.estimator_.predict(X)

    def predict(self, X):
        scores = self._get_scores(X)
        return (scores >= self.threshold).astype(int)

    def score(self, X, y):
        scores = self._get_scores(X)
        return roc_auc_score(y, scores)


# ----- Function that builds a pipeline with any linear model -----
def make_linear_pipeline(regressor):
    wrapped = ROCThresholdWrapper(regressor, threshold=0.5)
    pipe = Pipeline([
        ("preprocess", preprocess),
        ("model", wrapped),
    ])
    return pipe


# ----- Try different models -----
models = [
    # LinearRegression(),
    # Ridge(alpha=1.0),
    # Lasso(alpha=0.001),
    # ElasticNet(alpha=0.1),
    # Lars(),
    # LassoLars(),
    # OrthogonalMatchingPursuit(),
    # BayesianRidge(),
    # ARDRegression(),
    # HuberRegressor(),
    # RANSACRegressor(),
    # # TheilSenRegressor(),
    # # PoissonRegressor(),
    # # GammaRegressor(),
    # TweedieRegressor(power=1.5),
    # SGDRegressor(max_iter=1000, tol=1e-3),
    # # QuantileRegressor(quantile=0.5),
]

for model in models:
    pipe = make_linear_pipeline(model)
    pipe.fit(X_train, y_train)
    score = pipe.score(X_test, y_test)  # this is ROC AUC now
    print(f"{model.__class__.__name__}: ROC AUC = {score:.4f}")


### Best score for a linear model, no ensemble, on the selected test sample is 0.6931 ROC

# Non-Linear Models

In [ ]:
from sklearn.preprocessing import PolynomialFeatures, SplineTransformer
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestClassifier,
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    AdaBoostRegressor,
    BaggingRegressor,
    IsolationForest    # NOT a regressor, but included per request
)

# ----- Try different models -----
models = [

    # # ---- Polynomial Regression ----
    # Pipeline([
    #     ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    #     ("model", LinearRegression())
    # ]),

    # # ---- Spline Regression ----
    # Pipeline([
    #     ("spline", SplineTransformer(degree=3, n_knots=5)),
    #     ("model", LinearRegression())
    # ]),

    # # ---- Tree-Based Regressors ----
    # DecisionTreeRegressor(),
    # RandomForestRegressor(),
    # ExtraTreesRegressor(),
    # GradientBoostingRegressor(),
    # HistGradientBoostingRegressor(),
    # AdaBoostRegressor(),
    # BaggingRegressor(),

    # # # ---- Anomaly Detector (NOT a regressor) ----
    # # IsolationForest(),
]

for model in models:
    pipe = make_linear_pipeline(model)
    pipe.fit(X_train, y_train)
    score = pipe.score(X_test, y_test)  # this is ROC AUC now
    print(f"{model.__class__.__name__}: ROC AUC = {score:.4f}")

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

X, y = train_df.drop(columns=[target_col, "id"]), train_df[target_col]

preprocess = ColumnTransformer(
    transformers=[
        ("standard_scaler", StandardScaler(), list(X.columns)),
    ],
    remainder="passthrough",
)

def make_pipeline(regressor):
    return Pipeline([
        ("preprocess", preprocess),
        ("model", regressor)
    ])

chosen_model = RandomForestClassifier()
pipe = make_pipeline(chosen_model)
pipe.fit(X, y)
preds = pipe.predict_proba(test_df)[:, 1]

# Predictions and Submission

In [ ]:
sub_df = pd.read_csv(submission_csv_path)
sub_df[target_col] = preds
sub_df.to_csv('submission.csv', index=False)